# BIM RAG — Full Ingestion Pipeline (structured import + embeddings + 3D viewer artifact)

Runs the complete pipeline needed to make an IFC file fully usable by the app:

1. **Structured import + embeddings** — `ifc_to_db()` parses the IFC, upserts entities/relationships
   into PostgreSQL (scoped by `source_model_id`, derived from the file's SHA-256 fingerprint), enables
   pgvector, and generates/embeds `rag_documents` with `BAAI/bge-m3`.
2. **3D viewer artifact** — converts the same IFC to a That Open Fragments `.frag` file via the
   frontend's `npm run prepare:model` tool, written to `model_assets/{source_model_id}/{fingerprint}.frag`,
   which is what the backend serves at `GET /api/models/{source_model_id}/viewer-asset`.

Both steps are idempotent — safe to re-run on the same file (already-valid rows/artifacts are skipped).

In [1]:
import subprocess
import sys
import time
from pathlib import Path

import bim_rag
from bim_rag.pipeline_structured import ifc_to_db
from bim_rag.reporting import print_report

# ingestion/src/bim_rag/__init__.py -> parents[2] == ingestion/
INGESTION_ROOT = Path(bim_rag.__file__).resolve().parents[2]
REPO_ROOT = INGESTION_ROOT.parent
IFC_ORIGINAL_DIR = INGESTION_ROOT / "ifc_original"
FRONTEND_DIR = REPO_ROOT / "frontend"

## Pipeline function

One argument: the IFC file address. Accepts an absolute path, or just a filename that lives in
`ingestion/ifc_original/` (e.g. `"FOJAB_Landsarkivet.ifc"`).

In [2]:
def run_full_ingestion(ifc_file_address: str) -> dict:
    """Run the full ingestion pipeline for one IFC file.

    Performs, in order:
      1. Structured import (entities + relationships) into PostgreSQL.
      2. pgvector setup + BAAI/bge-m3 embeddings into rag_documents.
      3. IFC -> Fragments conversion for the 3D viewer (model_assets/{source_model_id}/*.frag).

    Args:
        ifc_file_address: Absolute path to an .ifc file, or a filename located
            in ingestion/ifc_original/.

    Returns:
        dict with the structured/embedding report and the resolved source_model_id.
    """
    ifc_path = Path(ifc_file_address)
    if not ifc_path.is_file():
        candidate = IFC_ORIGINAL_DIR / ifc_path.name
        if candidate.is_file():
            ifc_path = candidate
        else:
            raise FileNotFoundError(f"IFC source not found: {ifc_file_address}")

    print(f"=== Step 1/2: Structured import + embeddings ({ifc_path.name}) ===")
    t0 = time.time()
    db_report = ifc_to_db(str(ifc_path))
    print_report(db_report, label="Structured Import + Embedding Report")
    print(f"Step 1/2 elapsed: {time.time() - t0:.1f}s")

    source_model_id = db_report["source_model_id"]

    print(f"\n=== Step 2/2: 3D viewer artifact (model_id={source_model_id}) ===")
    t1 = time.time()
    cmd = f'npm run prepare:model -- --input "{ifc_path}" --model-id {source_model_id}'
    result = subprocess.run(
        cmd,
        cwd=str(FRONTEND_DIR),
        shell=True,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(
            f"3D artifact preparation failed (exit code {result.returncode}) "
            f"for model_id={source_model_id}"
        )
    print(f"Step 2/2 elapsed: {time.time() - t1:.1f}s")

    print("\n=== Ingestion pipeline complete ===")
    print(f"source_model_id : {source_model_id}")
    print(f"total_rag_docs  : {db_report['total_rag_docs']}")
    print(f"viewer asset    : model_assets/{source_model_id}/<fingerprint>.frag")

    return {"db_report": db_report, "source_model_id": source_model_id}

## Run it

To ingest a new IFC file, drop it in `ingestion/ifc_original/` and change the argument below to its
filename (or full path). After this cell finishes, the model is queryable and viewable in the app.

In [3]:
result = run_full_ingestion("FOJAB_Landsarkivet.ifc")

=== Step 1/2: Structured import + embeddings (FOJAB_Landsarkivet.ifc) ===
[ifc_to_db] Source: FOJAB_Landsarkivet.ifc
[ifc_to_db] Computing fingerprint...
[ifc_to_db] SHA-256: 9d0706a193190ac8...
[ifc_to_db] Scanning IFC model...


[ifc_to_db] Schema=IFC2X3  Total=3180316  Entities=20975  Relationships=19938
[ifc_to_db] Connecting to database...
[ifc_to_db] DB connection OK.
[ifc_to_db] Creating/verifying schema tables...
[ifc_to_db] Importing 20975 entities...
[ifc_to_db] Existing source model id=2


[ifc_to_db] Entities new=0  updated=20975  failures=0
[ifc_to_db] Importing 19938 relationships...


[ifc_to_db] Relationships 19938/19938...
[ifc_to_db] Rels new=0  updated=19938  failures=0
[ifc_to_db] Members total=66283  resolved=60296  unresolved=5987
[ifc_to_db] Structured import complete.


C:\Users\kdgki\anaconda3\envs\bim_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


...

[Stage 2] Entities to (re)generate: 0  skipped (valid): 20975

[Stage 2] Entity docs: new=0 updated=0 skipped_valid=20975 truncated=0 failures=0


[Stage 2] Preparing relationship_description docs for 19938 relationships...


[Stage 2] Relationships to (re)generate: 0  skipped (valid): 19938

[Stage 2] Rel docs: new=0 updated=0 skipped_valid=19938 truncated=0 failures=0
[ifc_to_db] Vector phase complete.



  Structured Import + Embedding Report
{
  "timestamp": "2026-07-17T19:03:46.097429+00:00",
  "source_model_id": 2,
  "file_fingerprint_prefix": "9d0706a193190ac8...",
  "ifc_schema": "IFC2X3",
  "total_ifc_entity_count": 3180316,
  "eligible_entity_count": 20975,
  "relationship_count_ifc": 19938,
  "relationship_class_counts": {
    "IfcRelVoidsElement": 2069,
    "IfcRelFillsElement": 873,
    "IfcRelDefinesByProperties": 6853,
    "IfcRelContainedInSpatialStructure": 527,
    "IfcRelConnectsPathElements": 2126,
    "IfcRelDefinesByType": 980,
    "IfcRelAssociatesMaterial": 5566,
    "IfcRelAggregates": 104,
    "IfcRelAssignsToGroup": 419,
    "IfcRelAssociatesClassification": 421
  },
  "entity_class_counts": {
    "IfcOpeningElement": 2069,
    "IfcPipeSegmentType": 3,
    "IfcFlowSegment": 3,
    "IfcPropertySet": 6853,
    "IfcRailing": 59,
    "IfcGroup": 474,
    "IfcFurnishingElement": 3440,
    "IfcFurnitureType": 315,
    "IfcWallStandardCase": 1929,
    "IfcBuildingElem


> bim-rag-frontend@0.1.0 prepare:model
> tsx scripts/prepare-viewer-model.ts --input C:\Users\kdgki\Desktop\MSCDP\Projects\BIM_RAG\ingestion\ifc_original\FOJAB_Landsarkivet.ifc --model-id 2

Reading IFC: FOJAB_Landsarkivet.ifc
  size 177.5 MB, sha256 9d0706a193190ac8520c104cdec20c4db073243de3478faeea93a887a405f2ab
Artifact already exists (14.9 MB): model_assets\2\9d0706a193190ac8520c104cdec20c4db073243de3478faeea93a887a405f2ab.frag
Use --force to reconvert. Nothing to do.

Step 2/2 elapsed: 1.2s

=== Ingestion pipeline complete ===
source_model_id : 2
total_rag_docs  : 40913
viewer asset    : model_assets/2/<fingerprint>.frag
